# Installing necessary libraries

In [1]:
! pip install langchain langchain-openai langchain-community langchain-text-splitters langchain-postgres

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.7/202.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.6 MB/s eta 0:00:00


# Making necessary imports

In [3]:
from langchain_openai.llms import OpenAI
from langchain.document_loaders import PyPDFLoader
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.runnables import chain
from langchain_postgres.vectorstores import PGVector
from langchain_core.documents import Document
import uuid

## Setting up the openAI key

In [ ]:
!export OPENAI_API_KEY=your-key

# Stage one:
### Loading and splitting the document

In [ ]:
raw_documents = PyPDFLoader("your_pdf_here.pdf").load()

# Splitting the document
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
splitted_docs = splitter.split_documents(raw_documents)

# Stage two:
### Building the embeddigs and VectorDB

In [ ]:
# Embedding the document
embd_model = OpenAIEmbeddings()
connection = "your_connection_string_to_PGvector"

db = PGVector.from_documents(splitted_docs, embd_model, connection)

# Stage three:
### Biulding up the Retrieving part

In [ ]:
retriever = db.as_retriever()

# Base prompt template
prompt = ChatPromptTemplate.from_template("""Answer the following question based on only the content below:
                                          {context}
                                          Question: {question}
                                          """)

llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

@chain
def QA(input):
  # Fetching relevant documnts
  relevant_docs = retriever.get_relevant_documents(input)
  # Foramatting the prompt
  formatted_prompt = prompt.invoke({"context": relevant_docs, "question": input})
  # Generate answer with the source document
  answer = llm.invoke(formatted_prompt)
  return {"Answer": answer, "Source": relevant_docs}

### To run :

In [ ]:
QA.invoke("your_prompt_here")